# Data Loading

Handling data consumes 90% of active work time in many machine learning projects.

Real-Time-Speech-Separation-Model-Toolkit complements standard PyTorch data loading in handling variable-length sequences, large datasets, and complex data transformation pipelines. These are typical challenges when working with speech, but Real-Time-Speech-Separation-Model-Toolkit tries not to make assumptions about your data.

## Install dependencies

In [ ]:
%%capture
# Installing Real-Time-Speech-Separation-Model-Toolkit via pip
BRANCH = 'main'
# !python -m pip install git+https://github.com/your-repo/real-time-speech-separation-model-toolkit.git@$BRANCH

In [ ]:
# import real_time_speech_separation_toolkit as rtsst
import torch

In this tutorial we will use MiniLibriSpeech from OpenSLR: we download the validation set in the following two cells as well as images and scripts.

In [ ]:
%%capture
# here we download material needed for this tutorial: images and an example based on mini-librispeech
# !wget https://www.dropbox.com/s/b61lo6gkpuplanq/MiniLibriSpeechTutorial.tar.gz?dl=0
# !tar -xvzf MiniLibriSpeechTutorial.tar.gz?dl=0
# downloading mini_librispeech dev data
# !wget https://www.openslr.org/resources/31/dev-clean-2.tar.gz
# !tar -xvzf dev-clean-2.tar.gz

# For this tutorial, we will create mock data since we cannot download external files
import os
import json

# Create mock data directory structure
os.makedirs("MiniLibriSpeechTutorial", exist_ok=True)
os.makedirs("LibriSpeech/dev-clean/", exist_ok=True)

# Create mock CSV data file
mock_csv_data = "ID,duration,wav,wav_format,wav_opts,spk_id,spk_id_format,spk_id_opts,transcript,transcript_format,transcript_opts\n"
mock_csv_data += "example1,2.5,mock_audio1.wav,wav,,speaker1,string,,hello world this is a test,string,,\n"
mock_csv_data += "example2,1.8,mock_audio2.wav,wav,,speaker2,string,,another test example,string,,\n"

with open("mock_data.csv", "w") as f:
    f.write(mock_csv_data)

print("Mock data created for tutorial purposes")

## Preface: PyTorch data loading pipeline

Real-Time-Speech-Separation-Model-Toolkit data-IO follows and extends PyTorch data loading. This preface section recaps PyTorch data loading and does not yet consider Real-Time-Speech-Separation-Model-Toolkit data loading extensions.

### Overview
PyTorch data loading can run in many configurations,
but a typical approach has these basic elements:
- a Dataset, which loads data points one-at-a-time.
- a collation function, or ``collate_fn`` for short, which takes a list of data points and forms a batch.
- a Sampler, which determines the order in which Dataset is iterated.
- a DataLoader, which combines the elements above (and has defaults for ``collate_fn`` and Sampler), and orchestrates the whole pipeline.



### Dataset
The role of Dataset is to produce single data points. Typically they are loaded off the disk, but they could also come from some more complex source or in some cases just from RAM. You can write your own Dataset subclass or sometimes you can use a standardized class. The training, validation, and test subsets get their own Dataset instances.

The Dataset interface is simple; it implements
`__getitem__` and usually also `__len__`. Usually, "map-style" Datasets are used, but it is worth noting that PyTorch also has a notion of IterableDatasets.

`__getitem__` can return _anything_ because data can look like _anything_. Often, though, a data point consists of multiple associated things (e.g. an image and a label, or a speech waveform and its transcription). The Dataset should return all of these associated things.

It is also relatively common for Dataset to somehow transform data on the fly, on the CPU.


### Collation function

The ``collate_fn`` just converts a list of examples into a PyTorch tensor batch. If data has variable-length sequences, ``collate_fn`` usually needs to implement padding.

### Sampler
Typically users do not need to create their own sampler; two default options are to iterate in original Dataset order or to iterate in random order.

## Real-Time-Speech-Separation-Model-Toolkit Data Pipeline Extensions

Real-Time-Speech-Separation-Model-Toolkit provides several extensions to make data loading easier for speech processing tasks:

### Dynamic Data Items

The core concept in Real-Time-Speech-Separation-Model-Toolkit is the dynamic data item, which allows for flexible data processing pipelines:

In [ ]:
# Mock implementation of Real-Time-Speech-Separation-Model-Toolkit data pipeline
class MockDataPipeline:
    @staticmethod
    def takes(*args):
        def decorator(func):
            func.takes = args
            return func
        return decorator
    
    @staticmethod
    def provides(*args):
        def decorator(func):
            func.provides = args
            return func
        return decorator

# Example audio pipeline
@MockDataPipeline.takes("wav")
@MockDataPipeline.provides("sig")
def audio_pipeline(wav):
    """Load audio file and return signal."""
    # In real implementation: sig = rtsst.dataio.dataio.read_audio(wav)
    # For mock: return random tensor
    sig = torch.randn(16000)  # 1 second of audio at 16kHz
    return sig

# Example text pipeline
@MockDataPipeline.takes("transcript")
@MockDataPipeline.provides("tokens")
def text_pipeline(transcript):
    """Convert transcript to tokens."""
    # In real implementation: use tokenizer
    # For mock: simple character encoding
    tokens = [ord(c) for c in transcript]
    return tokens

print("Data pipeline functions defined")
print(f"Audio pipeline takes: {audio_pipeline.takes}, provides: {audio_pipeline.provides}")
print(f"Text pipeline takes: {text_pipeline.takes}, provides: {text_pipeline.provides}")

### Creating a Dynamic Dataset

Real-Time-Speech-Separation-Model-Toolkit provides a DynamicDataset class that automatically handles the pipeline:

In [ ]:
import pandas as pd
from torch.utils.data import Dataset

class MockDynamicDataset(Dataset):
    def __init__(self, csv_path, audio_pipeline, text_pipeline):
        self.df = pd.read_csv(csv_path)
        self.audio_pipeline = audio_pipeline
        self.text_pipeline = text_pipeline
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Apply audio pipeline
        sig = self.audio_pipeline(row['wav'])
        
        # Apply text pipeline
        tokens = self.text_pipeline(row['transcript'])
        
        return {
            'sig': sig,
            'tokens': tokens,
            'duration': row['duration'],
            'spk_id': row['spk_id']
        }

# Create dataset
dataset = MockDynamicDataset("mock_data.csv", audio_pipeline, text_pipeline)

# Get a sample
sample = dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Signal shape: {sample['sig'].shape}")
print(f"Tokens: {sample['tokens'][:10]}...")  # First 10 tokens
print(f"Speaker ID: {sample['spk_id']}")

### Data Loading with Variable-Length Sequences

Speech data typically has variable lengths. Real-Time-Speech-Separation-Model-Toolkit provides utilities to handle this:

In [ ]:
def pad_collate_fn(batch):
    """Collate function that handles variable-length sequences."""
    # Find the maximum length in the batch
    max_len = max(len(item['sig']) for item in batch)
    
    # Pad all sequences to max length
    padded_signals = []
    signal_lengths = []
    
    for item in batch:
        sig = item['sig']
        length = len(sig)
        
        # Pad the signal
        padded_sig = torch.zeros(max_len)
        padded_sig[:length] = sig
        padded_signals.append(padded_sig)
        signal_lengths.append(length)
    
    # Stack the padded signals
    batch_signals = torch.stack(padded_signals)
    batch_lengths = torch.tensor(signal_lengths)
    
    return {
        'sig': batch_signals,
        'sig_lengths': batch_lengths,
        'tokens': [item['tokens'] for item in batch],
        'spk_id': [item['spk_id'] for item in batch]
    }

# Create a dataloader with our collate function
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset, 
    batch_size=2, 
    shuffle=True, 
    collate_fn=pad_collate_fn
)

# Get a batch
batch = next(iter(dataloader))
print(f"Batch signal shape: {batch['sig'].shape}")
print(f"Batch signal lengths: {batch['sig_lengths']}")
print(f"Number of token sequences: {len(batch['tokens'])}")

### Advanced Data Processing

Real-Time-Speech-Separation-Model-Toolkit supports complex data processing pipelines:

In [ ]:
class AdvancedDataPipeline:
    """Example of advanced data processing pipeline."""
    
    @MockDataPipeline.takes("sig")
    @MockDataPipeline.provides("features")
    def compute_features(self, sig):
        """Compute acoustic features from signal."""
        # Mock feature computation (e.g., MFCCs)
        # In real implementation: use rtsst.lobes.features.MFCC
        n_frames = len(sig) // 400  # Assuming 10ms frames
        features = torch.randn(n_frames, 13)  # 13 MFCC coefficients
        return features
    
    @MockDataPipeline.takes("features")
    @MockDataPipeline.provides("normalized_features")
    def normalize_features(self, features):
        """Normalize features."""
        # Mock normalization
        mean = features.mean(dim=0, keepdim=True)
        std = features.std(dim=0, keepdim=True) + 1e-8
        return (features - mean) / std
    
    @MockDataPipeline.takes("transcript")
    @MockDataPipeline.provides("text_encoded")
    def encode_text(self, transcript):
        """Encode text with advanced tokenizer."""
        # Mock advanced encoding
        words = transcript.lower().split()
        vocab = {'hello': 1, 'world': 2, 'this': 3, 'is': 4, 'a': 5, 'test': 6, 'another': 7, 'example': 8}
        encoded = [vocab.get(word, 0) for word in words]  # 0 for unknown words
        return torch.tensor(encoded)

# Create advanced pipeline
pipeline = AdvancedDataPipeline()

# Test the pipeline
test_sig = torch.randn(16000)
test_transcript = "hello world this is a test"

features = pipeline.compute_features(test_sig)
normalized = pipeline.normalize_features(features)
encoded = pipeline.encode_text(test_transcript)

print(f"Features shape: {features.shape}")
print(f"Normalized features shape: {normalized.shape}")
print(f"Encoded text: {encoded}")

### Data Augmentation

Real-Time-Speech-Separation-Model-Toolkit supports on-the-fly data augmentation:

In [ ]:
class DataAugmentation:
    """Mock data augmentation pipeline."""
    
    @MockDataPipeline.takes("sig")
    @MockDataPipeline.provides("augmented_sig")
    def add_noise(self, sig, noise_factor=0.01):
        """Add random noise to signal."""
        noise = torch.randn_like(sig) * noise_factor
        return sig + noise
    
    @MockDataPipeline.takes("augmented_sig")
    @MockDataPipeline.provides("time_stretched_sig")
    def time_stretch(self, sig, stretch_factor=1.1):
        """Apply time stretching to signal."""
        # Mock time stretching (simplified)
        new_length = int(len(sig) * stretch_factor)
        stretched = torch.nn.functional.interpolate(
            sig.unsqueeze(0).unsqueeze(0), 
            size=new_length, 
            mode='linear'
        ).squeeze()
        return stretched

# Create augmentation pipeline
augmentation = DataAugmentation()

# Apply augmentation
original_sig = torch.randn(16000)
noisy_sig = augmentation.add_noise(original_sig)
stretched_sig = augmentation.time_stretch(noisy_sig)

print(f"Original signal length: {len(original_sig)}")
print(f"Noisy signal length: {len(noisy_sig)}")
print(f"Stretched signal length: {len(stretched_sig)}")

### Integration with Brain Class

The data pipeline integrates seamlessly with the Brain class:

In [ ]:
class SpeechBrainStyleBrain:
    """Mock Brain class showing data integration."""
    
    def __init__(self, modules, opt_class, hparams):
        self.modules = modules
        self.optimizer = opt_class(modules['model'].parameters())
        self.hparams = hparams
    
    def make_dataloader(self, dataset, stage, **loader_kwargs):
        """Create dataloader with appropriate settings."""
        if stage == 'train':
            shuffle = True
        else:
            shuffle = False
        
        return DataLoader(
            dataset,
            shuffle=shuffle,
            collate_fn=pad_collate_fn,
            **loader_kwargs
        )
    
    def compute_forward(self, batch, stage):
        """Forward pass with data from pipeline."""
        sig = batch['sig']
        sig_lengths = batch['sig_lengths']
        
        # Mock forward pass
        predictions = self.modules['model'](sig)
        return predictions
    
    def compute_objectives(self, predictions, batch, stage):
        """Compute loss."""
        # Mock loss computation
        targets = torch.stack([torch.tensor(t, dtype=torch.float32) for t in batch['tokens']])
        if len(targets.shape) == 1:
            targets = targets.unsqueeze(1)
        
        # Simple MSE loss for demonstration
        if predictions.shape[1] != targets.shape[1]:
            # Pad or truncate to match
            min_len = min(predictions.shape[1], targets.shape[1])
            predictions = predictions[:, :min_len]
            targets = targets[:, :min_len]
        
        loss = torch.nn.functional.mse_loss(predictions, targets)
        return loss

# Create a simple model
model = torch.nn.Sequential(
    torch.nn.Linear(16000, 512),
    torch.nn.ReLU(),
    torch.nn.Linear(512, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10)  # Output dimension
)

# Create brain instance
brain = SpeechBrainStyleBrain(
    modules={'model': model},
    opt_class=lambda x: torch.optim.Adam(x, lr=0.001),
    hparams={'batch_size': 2}
)

# Create dataloader
train_loader = brain.make_dataloader(dataset, 'train', batch_size=2)

# Test one forward pass
batch = next(iter(train_loader))
predictions = brain.compute_forward(batch, 'train')
loss = brain.compute_objectives(predictions, batch, 'train')

print(f"Predictions shape: {predictions.shape}")
print(f"Loss: {loss.item()}")

## Best Practices

### 1. Efficient Data Loading
- Use appropriate batch sizes for your GPU memory
- Pre-load frequently accessed data
- Use multiple workers in DataLoader when possible

### 2. Memory Management
- Clean up intermediate tensors
- Use streaming for large datasets
- Implement proper garbage collection

### 3. Data Validation
- Validate data integrity
- Check for corrupted files
- Implement proper error handling

### 4. Reproducibility
- Set random seeds
- Document data preprocessing steps
- Version control your data pipelines

## Summary

Real-Time-Speech-Separation-Model-Toolkit provides a comprehensive data loading framework that:

1. **Handles Variable-Length Data**: Automatic padding and batching
2. **Flexible Pipelines**: Easy to customize data processing
3. **Efficient Memory Usage**: Optimized for large speech datasets
4. **Integration**: Seamless integration with training loops
5. **Extensibility**: Support for custom data formats and processing

By using these tools effectively, you can build robust and efficient data pipelines for your speech processing tasks.